# Datathon 2026 Task 1 — Ridge inference

Self-contained inference for audited candidate `d1-e002-ridge`. It uses only official competition inputs, NumPy, and pandas; no API, credentials, external data, or pretrained weights.

In [ ]:
import hashlib
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

HISTORY_LENGTH = 15
HORIZONS = np.array([5, 10, 15], dtype=np.int64)
ROAD_COUNT = 1260
ALPHA = 0.1
CHUNK_SIZE = 256
started = time.perf_counter()

def is_data_root(path):
    path = Path(path)
    return (
        (path / 'train').is_dir()
        and (path / 'test' / 'test_X_hist.npy').is_file()
        and (path / 'sample_submission.csv').is_file()
    )

override = os.environ.get('DATATHON_DATA_ROOT')
if override:
    data_root = Path(override)
    if not is_data_root(data_root):
        raise FileNotFoundError(f'DATATHON_DATA_ROOT is not a valid dataset root: {data_root}')
else:
    kaggle_input = Path('/kaggle/input')
    candidates = sorted({
        path.parent
        for path in kaggle_input.rglob('sample_submission.csv')
        if is_data_root(path.parent)
    })
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one Task 1 dataset under /kaggle/input, found {candidates}')
    data_root = candidates[0]

output_path = Path(os.environ.get('DATATHON_OUTPUT_PATH', '/kaggle/working/submission.csv'))
print(f'data_root={data_root}')
print(f'output_path={output_path}')


In [ ]:
def windows_at_origins(block, origins):
    origins = np.asarray(origins, dtype=np.int64)
    offsets = np.arange(HISTORY_LENGTH - 1, -1, -1, dtype=np.int64)
    histories = np.asarray(block[origins[:, None] - offsets[None, :]], dtype=np.float32)
    targets = np.asarray(block[origins[:, None] + HORIZONS[None, :]], dtype=np.float32)
    return histories, targets

def history_features(history):
    history = np.asarray(history, dtype=np.float32)
    if history.ndim != 3 or history.shape[1] != HISTORY_LENGTH:
        raise ValueError(f'Expected (samples, 15, roads), got {history.shape}')
    recent = history[:, -5:]
    x = np.arange(5, dtype=np.float32)
    centered = x - x.mean()
    slope5 = np.einsum('str,t->sr', recent, centered) / float(np.square(centered).sum())
    return np.stack((
        history[:, -1],
        history[:, -3:].mean(axis=1),
        recent.mean(axis=1),
        history.mean(axis=1),
        slope5,
    ), axis=2).astype(np.float32, copy=False)

@dataclass
class RoadRidgeModel:
    feature_mean: np.ndarray
    feature_scale: np.ndarray
    target_mean: np.ndarray
    coefficients: np.ndarray

    def predict(self, history):
        features = history_features(history).astype(np.float64)
        standardized = (features - self.feature_mean[None, :, :]) / self.feature_scale[None, :, :]
        prediction = self.target_mean[None, :, :] + np.einsum('nrf,rfh->nrh', standardized, self.coefficients)
        prediction = prediction.transpose(0, 2, 1)
        zero_history = np.all(np.asarray(history) == 0, axis=1)
        prediction = np.where(zero_history[:, None, :], 0.0, prediction)
        return np.maximum(prediction, 0.0).astype(np.float32)

def fit_road_ridge(block, origin_start, origin_end, alpha=ALPHA, chunk_size=CHUNK_SIZE):
    road_count = block.shape[1]
    feature_count = 5
    horizon_count = len(HORIZONS)
    sum_x = np.zeros((road_count, feature_count), dtype=np.float64)
    sum_y = np.zeros((road_count, horizon_count), dtype=np.float64)
    sum_xx = np.zeros((road_count, feature_count, feature_count), dtype=np.float64)
    sum_xy = np.zeros((road_count, feature_count, horizon_count), dtype=np.float64)
    sample_count = 0
    for start in range(origin_start, origin_end + 1, chunk_size):
        stop = min(start + chunk_size, origin_end + 1)
        histories, targets = windows_at_origins(block, np.arange(start, stop))
        x = history_features(histories).astype(np.float64)
        y = targets.transpose(0, 2, 1).astype(np.float64)
        sum_x += x.sum(axis=0)
        sum_y += y.sum(axis=0)
        sum_xx += np.einsum('nrf,nrg->rfg', x, x, optimize=True)
        sum_xy += np.einsum('nrf,nrh->rfh', x, y, optimize=True)
        sample_count += len(x)
    feature_mean = sum_x / sample_count
    target_mean = sum_y / sample_count
    covariance = sum_xx / sample_count - np.einsum('rf,rg->rfg', feature_mean, feature_mean)
    cross_covariance = sum_xy / sample_count - np.einsum('rf,rh->rfh', feature_mean, target_mean)
    variance = np.maximum(np.diagonal(covariance, axis1=1, axis2=2), 0.0)
    feature_scale = np.sqrt(variance)
    feature_scale = np.where(feature_scale > 1e-6, feature_scale, 1.0)
    standardized_covariance = covariance / (feature_scale[:, :, None] * feature_scale[:, None, :])
    standardized_cross_covariance = cross_covariance / feature_scale[:, :, None]
    system = standardized_covariance + alpha * np.eye(feature_count)[None, :, :]
    coefficients = np.linalg.solve(system, standardized_cross_covariance)
    return RoadRidgeModel(feature_mean, feature_scale, target_mean, coefficients)

def classify_test_regime(history, zero_road_threshold=100):
    zero_road_count = np.all(np.asarray(history) == 0, axis=1).sum(axis=1)
    return (zero_road_count > zero_road_threshold).astype(np.int64)


In [ ]:
train_paths = sorted((data_root / 'train').glob('train_speed_*.npy'))
if len(train_paths) != 2:
    raise ValueError(f'Expected two train speed blocks, found {train_paths}')
blocks = [np.load(path, mmap_mode='r') for path in train_paths]
test_history = np.load(data_root / 'test' / 'test_X_hist.npy', mmap_mode='r')
if test_history.shape != (540, 15, ROAD_COUNT):
    raise ValueError(f'Unexpected test shape: {test_history.shape}')
models = [
    fit_road_ridge(block, 14, len(block) - int(HORIZONS.max()) - 1)
    for block in blocks
]
regimes = classify_test_regime(test_history)
if [int((regimes == index).sum()) for index in range(2)] != [372, 168]:
    raise ValueError('Unexpected test regime counts')
prediction = np.empty((len(test_history), len(HORIZONS), ROAD_COUNT), dtype=np.float32)
for regime_index, model in enumerate(models):
    mask = regimes == regime_index
    prediction[mask] = model.predict(test_history[mask])
if not np.isfinite(prediction).all() or (prediction < 0).any():
    raise ValueError('Predictions must be finite and nonnegative')
zero_history = np.all(test_history == 0, axis=1)
if not (prediction.transpose(0, 2, 1)[zero_history] == 0).all():
    raise ValueError('Zero-history guard failed')


In [ ]:
template = pd.read_csv(data_root / 'sample_submission.csv')
if template.columns.tolist() != ['id', 'speed']:
    raise ValueError(f'Unexpected template columns: {template.columns.tolist()}')
expected_rows = prediction.size
if len(template) != expected_rows:
    raise ValueError(f'Template has {len(template)} rows, expected {expected_rows}')
expected_ids = [
    f'test_{sample:05d}_h{int(horizon)}_r{road}'
    for sample in range(prediction.shape[0])
    for horizon in HORIZONS
    for road in range(prediction.shape[2])
]
if template['id'].tolist() != expected_ids:
    raise ValueError('Template ID order does not match sample-horizon-road order')
template['speed'] = prediction.reshape(-1)
output_path.parent.mkdir(parents=True, exist_ok=True)
template.to_csv(output_path, index=False)
summary = {
    'experiment_id': 'd1-e002-ridge',
    'rows': int(prediction.size),
    'regime_counts': [int((regimes == index).sum()) for index in range(2)],
    'zero_history_pairs': int(zero_history.sum()),
    'min': float(prediction.min()),
    'max': float(prediction.max()),
    'mean': float(prediction.mean()),
    'float32_prediction_sha256': hashlib.sha256(prediction.astype('<f4', copy=False).tobytes()).hexdigest(),
    'runtime_seconds': float(time.perf_counter() - started),
    'output': str(output_path),
}
print(json.dumps(summary, indent=2))
